In [1]:
from espn_api.football import League, Player
import json
import espn_api
import os
from itertools import groupby
from collections import defaultdict
import espn_api.football as e
import requests
import pandas as pd

from typing import List, Dict
import pyperclip
import time

from infra import *

In [2]:
POSITIONS = ['WR', 'RB', 'TE', 'DE', 'LB', 'S', 'CB', 'DT', 'QB', 'K']

GAME_LOGS_LINK = "https://site.web.api.espn.com/apis/common/v3/sports/football/nfl/athletes/{player}/gamelog?region=us&lang=en&contentorigin=espn&season={year}"

In [3]:
LAMAR = 3916387
DICKER = 4362081

In [11]:
league = League(league_id=62847108, year=2024, debug = False) # change to dict
teams = league.teams
player_ids = [key for key in league.player_map if type(key) == int]

## Data description

Raw season stats: each player's stats for each game week for each scoring category. One file per season. List of jsons

Clean season stats: Converted into csv file with player id, season, week, position. Once csv file per season

Season appearences: player id, team id, opponent id, season, week, score

Lineups: Each fantasy team's roster with position, injury status, active status etc


## Get all players and write to file

In [8]:
def get_players_batch(league: League, ids: int):
    players = []
    player_infos = league.player_info(playerId = ids)
    for player in player_infos:
        team = player.proTeam
        player_stats = dict()
        player_stats["id"] = player.playerId
        player_stats["name"] = player.name
        player_stats["position"] = player.position
        player_stats["stats"] = dict()
        for week in player.stats.keys():
            player.stats[week]["team"] = team
            #if (week < 0) and (week in schedule.keys()) and (team in schedule[week].keys()):
            #    player.stats[week]["opponent"] = schedule[week][team]
            #else:
            #    player.stats[week]["opponent"] = ""
            player_stats["stats"][week] = player.stats[week]
        players.append(player_stats)
    
    return players


def get_all_players(league: League, ids: List[int]):

    start = 0
    batch_size = 1_000

    players = []

    while start < len(ids):
        time.sleep(3)
        players += get_players_batch(league, ids[start:start+batch_size])
        start += batch_size
    
    write_raw_season_stats(league.year, players)


for season in range(2021, 2025):
    
    league = League(league_id=62847108, year=season, debug = False)
    
    player_ids = [key for key in league.player_map if type(key) == int]
    
    get_all_players(league, player_ids)

## Clean Stats

In [42]:
def create(player, actual):
    key = "breakdown" if actual else "projected_breakdown" # only exists for season total
    rows = []
    
    for week in player["stats"]:

        if actual and week == "0":
             continue

        if not actual and week != "0":
            continue
            
        stats = dict()
        stats["player_id"] = player["id"]
        stats["week"] = week
        # stats["position"] = player["position"]

        for k in player["stats"][week][key]:
                stats[k] = player["stats"][week][key][k]
                
        rows.append(stats)

    return rows


def clean_season_stats(season: int):

    actual_stats = read_raw_season_stats(2024)
    actual_rows = []
    projected_rows = []
    for player in actual_stats:
        actual_rows += create(player, True)
        # projected_rows += create(player, False)


    actual_stats = pd.DataFrame(actual_rows)
    actual_stats.fillna(0, inplace=True)

    projected_stats = pd.DataFrame(actual_rows)
    projected_stats.fillna(0, inplace=True)

    write_clean_season_stats(season, actual_stats)
    write_clean_projected_season_stats(season, projected_stats)

for season in range(2021, 2025):
    clean_season_stats(season)

## Calculate points

In [13]:
points_by_action = pd.read_csv("constants/scoring_format.csv").set_index("lookupName")
points_by_action_series = points_by_action.points

In [430]:
def f(row):
    return (points_by_action.points * row[row.index.isin(points_by_action.index)].astype(float)).sum()

# stats["calculated"] = stats.apply(f, 1)

In [14]:
def get_stat_types(stats):
    stat_types = set()
    for player in stats:
        for week in player["stats"]:
            for stat_type in player["stats"][week]["breakdown"]:
                if not stat_type.isnumeric():
                    stat_types.add(stat_type)
    return stat_types

In [18]:
len(player_ids)

2707

In [12]:
def parse_gamelog(gamelog: dict, season: int, player_id: int) -> dict:
    
    opponent = gamelog["opponent"]["id"]
    is_home = gamelog["homeTeamId"] != opponent
    team = gamelog["homeTeamId"] if is_home else gamelog["awayTeamId"]
    score = gamelog["homeTeamScore"] if is_home else gamelog["awayTeamScore"]
    opponent_score = gamelog["homeTeamScore"] if not is_home else gamelog["awayTeamScore"]
    date = gamelog["gameDate"]
    game_id = gamelog["id"]
    week = gamelog["week"]

    return {
        "player_id": player_id,
        "game_id": game_id,
        "season": season,
        "week": week,
        "team": team,
        "opponent": opponent,
        "date": date,
        "team_score": score,
        "opponent_score": opponent_score,
    }


def parse_season(season_logs, season, player):
    game_logs = []

    if "events" not in season_logs.keys():
        return game_logs
    
    for game_id in season_logs["events"].keys():
        game_logs.append(parse_gamelog(season_logs["events"][game_id], season, player))

    return game_logs


def get_season_appearences(player_ids, start_year, end_year):

    seasons = range(start_year, end_year+1)
    logs_by_season = {season: [] for season in seasons}

    for i, player in enumerate(player_ids):
        
        if (i % 100 == 0 & i > 0):
            time.sleep(300)
            print(i)
        
        time.sleep(0.5)
        
        print(i, player)
        for season in seasons:
            local_logs = try_read_raw_game_logs(player, season)
            if local_logs:
                season_logs = local_logs
            else:
                season_logs = requests.get(GAME_LOGS_LINK.format(player=player, year=season)).content
                season_logs = json.loads(season_logs)
                write_raw_game_logs(season_logs, player, season)
            logs_by_season[season] += parse_season(season_logs, season, player)

    for season in seasons:
        df = pd.DataFrame(logs_by_season[season])
        write_season_appearences(df, season)


get_season_appearences(player_ids, 2024, 2024)

0 4242896
1 4242899
2 3128747
3 3915189
4 4259300
5 4259299
6 15782
7 4259305
8 4259308
9 3915171
10 3128724
11 15795
12 3128721
13 3915165
14 3128720
15 3128715
16 3128746
17 4259324
18 3128745
19 15807
20 15810
21 15818
22 4685201
23 15826
24 15827
25 15830
26 3128815
27 15835
28 4062711
29 15839
30 15841
31 15842
32 15846
33 15847
34 15851
35 4685232
36 3915239
37 4259252
38 15861
39 15863
40 15864
41 4242873
42 4570561
43 4685247
44 15875
45 4373956
46 15880
47 15887
48 15920
49 15928
50 3915297
51 3915285
52 15936
53 4046353
54 4374148
55 3915282
56 3915398
57 15948
58 3915396
59 4374033
60 3915399
61 4374037
62 15958
63 15960
64 4374045
65 15964
66 3915381
67 15965
68 4242973
69 4242975
70 15971
71 3915373
72 4259370
73 15979
74 4570672
75 4570674
76 4374066
77 4242996
78 4243003
79 4243004
80 4243009
81 16002
82 4046536
83 4046537
84 4046532
85 16019
86 4046528
87 4046530
88 4046523
89 4243160
90 4259546
91 4046525
92 4259545
93 4259547
94 4046522
95 4259553
96 3915437
97 425955

In [23]:
def get_player_age_and_debut_year(id):
    url = f"http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/athletes/{id}"
    content = requests.get(url).content
    player_dict = json.loads(content)
    return player_dict["age"], player_dict["debutYear"]

# get_player_age_and_debut_year(3045282)

In [6]:
def get_player_lineup_info(player, week):
    info = {
        #'acquisitionType': player.acquisitionType,
        'week': week,
        'active_status': player.active_status,
        'avg_points': player.avg_points,
        #'eligibleSlots': player.eligibleSlots,
        #'game_date': player.game_date,
        #'game_played': player.game_played,
        'injured': player.injured,
        'injury_status': player.injuryStatus,
        'lineup_slot': player.lineupSlot,
        'name': player.name,
        'team_id': player.onTeamId,
        'on_bye_week': player.on_bye_week,
        #'percent_owned': player.percent_owned,
        #'percent_started': player.percent_started,
        'player_id': player.playerId,
        'points': player.points,
        #'points_breakdown': player.points_breakdown,
        #'pos_rank': player.posRank,
        'position': player.position,
        #'proTeam': player.proTeam,
        #'pro_opponent': player.pro_opponent,
        #'pro_pos_rank': player.pro_pos_rank,
        #'projected_avg_points': player.projected_avg_points,
        #'projected_breakdown': player.projected_breakdown,
        #'projected_points': player.projected_points,
        #'projected_total_points': player.projected_total_points,
        #'schedule': player.schedule,
        'slot_position': player.slot_position,
        #'stats': player.stats,
        #'total_points': player.total_points,
    }

    return info

In [9]:
def get_match_lineup(match, week):
    return [get_player_lineup_info(p, week) for p in match.home_lineup + match.away_lineup]

def get_lineups(league, min_week, max_week):
    for week in range(min_week, max_week+1):
        player_info = []
        boxscores = league.box_scores(week)
        
        for match in boxscores:
            player_info += get_match_lineup(match, week)
        
        write_lineups(pd.DataFrame(player_info), league.year, week)

get_lineups(league, 1, 7)

In [8]:
teams = []
for team in league.teams:
    teams.append({"id": team.team_id, "name": team.team_name})

pd.DataFrame(teams).to_csv("constants/teams.csv")